<a href="https://colab.research.google.com/github/GoldenEagle3k1/Filter-And-Summarizer-Using-Flask-And-Spark/blob/main/Filter%26Summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from flask import Flask, jsonify, request
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

app = Flask(__name__)

print("Initializing PySpark...")
# ---------------------------------------------------------
# 1. Initialize PySpark Session & Load Data
# ---------------------------------------------------------
spark = SparkSession.builder \
    .appName("Student_Performance_BigData") \
    .master("local[*]") \
    .getOrCreate()

# Load the CSV file uploaded to Colab
df = spark.read.csv("maths1.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("students")

# ---------------------------------------------------------
# 2. Train Spark MLlib Model
# ---------------------------------------------------------
print("Training MLlib Linear Regression Model...")
feature_columns = ["studytime", "failures", "absences", "G1", "G2"]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
ml_data = assembler.transform(df)

lr = LinearRegression(featuresCol="features", labelCol="G3")
model = lr.fit(ml_data)
print(f"Model trained successfully! (R2 Score: {round(model.summary.r2, 2)})")

# ---------------------------------------------------------
# 3. Flask API Endpoints
# ---------------------------------------------------------
@app.route('/', methods=['GET'])
def home():
    return "API is running! Add /api/students/summary to the URL to see data."

@app.route('/api/students/summary', methods=['GET'])
def get_summary():
    result_df = spark.sql("""
        SELECT studytime, COUNT(*) as total_students, ROUND(AVG(G3), 2) as avg_final_grade
        FROM students GROUP BY studytime ORDER BY studytime ASC
    """)
    return jsonify({"status": "success", "data": result_df.toPandas().to_dict(orient="records")})

@app.route('/api/students/filter', methods=['GET'])
def filter_data():
    has_internet = request.args.get('internet', 'yes')
    gender = request.args.get('sex', 'F')
    filtered_df = df.filter((col("internet") == has_internet) & (col("sex") == gender))
    clean_df = filtered_df.select("school", "sex", "age", "internet", "studytime", "G3")
    return jsonify({"status": "success", "data": clean_df.toPandas().to_dict(orient="records")})

@app.route('/api/predict', methods=['POST'])
def predict_score():
    req_data = request.get_json()
    try:
        st_time = float(req_data.get('studytime', 2))
        fails = float(req_data.get('failures', 0))
        absen = float(req_data.get('absences', 0))
        g1 = float(req_data.get('G1', 10))
        g2 = float(req_data.get('G2', 10))

        input_data = spark.createDataFrame([(st_time, fails, absen, g1, g2)], feature_columns)
        input_vector = assembler.transform(input_data)

        prediction_df = model.transform(input_vector)
        predicted_value = prediction_df.select("prediction").collect()[0][0]
        final_prediction = max(0, min(20, predicted_value))

        return jsonify({"status": "success", "predicted_G3_grade": round(final_prediction, 2)})
    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 400

# ---------------------------------------------------------
# 4. Colab Execution Logic
# ---------------------------------------------------------
if __name__ == '__main__':
    from google.colab.output import eval_js
    # Generate the public Colab link
    colab_url = eval_js("google.colab.kernel.proxyPort(5000)")
    print("\n" + "="*50)
    print("✅ YOUR API IS READY!")
    print(f"👉 CLICK HERE TO OPEN: {colab_url}")
    print("="*50 + "\n")

    # Run Flask server
    app.run(host='0.0.0.0', port=5000)

Initializing PySpark...
Training MLlib Linear Regression Model...
Model trained successfully! (R2 Score: 0.83)

✅ YOUR API IS READY!
👉 CLICK HERE TO OPEN: https://5000-m-s-kkb-usw3b0-207l1qtmkmyuu-b.us-west3-0.prod.colab.dev

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [08/May/2026 19:50:36] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/May/2026 19:50:36] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [08/May/2026 19:51:06] "GET /api/students/summary HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/May/2026 19:53:59] "GET /api/students/filter?internet=yes&sex=M HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/May/2026 19:57:11] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/May/2026 20:02:27] "GET /api/predict HTTP/1.1" 405 -
